# Production Line

Sends a work order to a Line Supervisor, which starts the line. Each stage (Cut, Weld, Paint, Inspection) does its work and hands off to the next, then Shipping returns the result. Everything runs on this machine with the Perceivo SDK. No model and no key are needed.

Open this notebook from this folder. Each stage waits `seconds_per_stage`, so the run takes about 20 seconds.

## Install the SDK

Installs perceivo 1.0.0 or later into this notebook's kernel. If it upgrades an older copy, restart the kernel before you go on.

In [1]:
%pip install -q "perceivo>=1.0.0"

Note: you may need to restart the kernel to use updated packages.


## Check the version

Confirms this notebook is using perceivo 1.0.0 or later. If you already had an older version, the cell above upgraded it, but Jupyter keeps the old one loaded until you restart the kernel. In that case this cell stops and tells you to restart.

In [2]:
import perceivo

if int(perceivo.__version__.split(".")[0]) < 1:
    raise RuntimeError(
        f"This example needs perceivo 1.0.0 or later, but this kernel has perceivo {perceivo.__version__} "
        f"from {perceivo.__file__}. Install perceivo 1.0.0 or later (the cell above does this), restart the kernel, then run all cells."
    )

## Load the request

`input.json` holds the sample request. Edit it to try your own.

In [3]:
import json

import perceivo.orchestration

request = json.load(open("input.json", encoding="utf-8"))
print(json.dumps(request, indent=2))

{
  "product": "bicycle frame",
  "seconds_per_stage": 5
}


## Run the orchestration

Runs every step on this machine. The run logs print below this cell as it goes: one `[INFO]` line as each step starts, plus anything the step logs. For more detail, set `level: DEBUG` under `logging` in `perceivo.yaml`. If a step fails, the cell stops with the error.

In [4]:
result = await perceivo.orchestration.run(".", input_data=request, config_path="perceivo.yaml")
if not result.success:
    raise RuntimeError(f"The run failed: {result.error}")
print("success:", result.success)

[INFO] Compiled orchestration: 7 nodes, start=node_work_order, root=node_line_supervisor, end=node_shipping
[INFO] perceivo SDK connected (local development mode)
[INFO] Loaded component: work_order from work-order.py
[INFO] Loaded component: line_supervisor from line-supervisor.py
[INFO] Loaded component: cut from cut.py
[INFO] Loaded component: weld from weld.py
[INFO] Loaded component: paint from paint.py
[INFO] Loaded component: inspection from inspection.py
[INFO] Loaded component: shipping from shipping.py
[INFO] Executing: Work Order (START)
[INFO] Executing: Line Supervisor (AGENT)
[INFO] Executing: Cut (TOOL)
[INFO] Line Supervisor: starting the line  {'product': 'bicycle frame'}
[INFO] Cut: cutting the parts  {'product': 'bicycle frame', 'seconds': 5}
[INFO] Executing: Weld (TOOL)
[INFO] Weld: welding the frame  {'product': 'bicycle frame', 'seconds': 5}
[INFO] Executing: Paint (TOOL)
[INFO] Paint: painting the frame  {'product': 'bicycle frame', 'seconds': 5}
[INFO] Executin

## Read the result

The **response data** is what Shipping returned. The **artifact** is the traveler that each stage signed.

In [5]:
print(json.dumps(result.response_data, indent=2))
print(result.artifact.data)

{
  "product": "bicycle frame",
  "stages_completed": 4,
  "status": "shipped"
}
bicycle frame
Cut: done
Weld: done
Paint: done
Inspection: passed



## Review the steps

The trace records each step in the order it finished, with its type and status. Each stage finishes only after the stages it handed off to, so they appear in reverse: Inspection, Paint, Weld, Cut.

In [6]:
for step in result.trace:
    print(f"{step.node:<16} {step.component_type:<6} {step.status}")

Work Order       START  ok
Inspection       TOOL   ok
Paint            TOOL   ok
Weld             TOOL   ok
Cut              TOOL   ok
Line Supervisor  AGENT  ok
Shipping         END    ok
